# Data Extraction
The sales files are arranged strangely as usual, with multiple files per year, and sometime with checkpoints.  
I'm going to follow the same strategy as Gwinnett, where I just take all of the sales data and deduplicate at the end.  

**Selected Data Files**  
These were all taken from the Dekalb-Hicks folder of the County Tax Assessment Data Folder  
Dekalb Count Full File2015/SALES.CSV  
Dekalb Count Full File EOY2015/SALES.CSV  
Dekalb Count Full File EOY2016/SALES.CSV  
Dekalb Count Full File EOY2017/SALES.CSV  
Dekalb Count Full File Oct 2018/SALES.CSV  
Dekalb Count Full File EOY2019/SALES.CSV  
Dekalb Count Full File EOY2020/SALES.CSV  
Dekalb Count Full File Aug 2021/SALES.CSV  

They were all renamed to the format SALES_20{XX}.CSV for convenience

In [2]:
import pandas as pd
import os

DATA_PATH = "../../data/dekalb"
OUT_PATH = "../../data/dekalb/out"

In [21]:
year_dfs = []

for file_p in os.listdir(DATA_PATH):
    if file_p.endswith(".CSV"):
        year = int(file_p.split("_")[1].split(".")[0])
        df = pd.read_csv(os.path.join(DATA_PATH, file_p))

        year_dfs.append((year, df))

In [22]:
year_dfs.sort(key = lambda x : x[0])

In [23]:
for year, df in year_dfs:
    print(year)
    print(df.columns)

2015
Index(['PARID', ' BOOK', ' PAGE', ' OLDOWN', ' OWN1', ' SALEDT', ' PRICE',
       ' SALETYPE', ' INSTRTYP', ' SALEVALDESCR'],
      dtype='object')
2016
Index(['PARID', ' BOOK', ' PAGE', ' OLDOWN', ' OWN1', ' SALEDT', ' PRICE',
       ' SALETYPE', ' INSTRTYP', ' SALEVALDESCR'],
      dtype='object')
2017
Index(['PARID', ' BOOK', ' PAGE', ' OLDOWN', ' OWN1', ' SALEDT', ' PRICE',
       ' SALETYPE', ' INSTRTYP', ' SALEVALDESCR'],
      dtype='object')
2018
Index(['PARID', ' BOOK', ' PAGE', ' OLDOWN', ' OWN1', ' SALEDT', ' PRICE',
       ' SALETYPE', ' INSTRTYP', ' SALEVALDESCR'],
      dtype='object')
2019
Index(['PARID', ' BOOK', ' PAGE', ' OLDOWN', ' OWN1', ' SALEDT', ' PRICE',
       ' SALETYPE', ' INSTRTYP', ' SALEVALDESCR'],
      dtype='object')
2020
Index(['PARID', ' BOOK', ' PAGE', ' OLDOWN', ' OWN1', ' SALEDT', ' PRICE',
       ' SALETYPE', ' INSTRTYP', ' SALEVALDESCR'],
      dtype='object')
2021
Index(['PARID', ' BOOK', ' PAGE', ' OLDOWN', ' OWN1', ' SALEDT', ' PRI

Luckily looks like the entire thing has the same data format.

In [25]:
year_df_list = []
for year, year_df in year_dfs:
    year_df = year_df.rename(columns={old: new for old, new in zip(year_df.columns, year_df.columns.str.strip())})

    for column in year_df.columns:
        if year_df.dtypes[column] == object:
            year_df[column] = year_df[column].str.strip()
    
    year_df['SALEDT'] = pd.to_datetime(year_df['SALEDT'], format="%d-%b-%y" ,errors="coerce")
    year_df_list.append(year_df)

total_digest_df = pd.concat(year_df_list)

In [26]:
old_rows = total_digest_df.shape[0]
total_digest_df_dedup = total_digest_df.drop_duplicates(subset=['PARID', 'BOOK', 'PAGE', 'SALEDT'])

new_rows = total_digest_df_dedup.shape[0]

print(f"{old_rows - new_rows} rows removed. {new_rows} Remaining.")

85 rows removed. 112149 Remaining.


In [29]:
total_digest_df_dedup = total_digest_df_dedup.sort_values(by = "SALEDT")

In [30]:
total_digest_df_dedup.to_csv(os.path.join(OUT_PATH, "DEKALB_SALES_FINAL.csv"), index=False)